# Multi-Head ConvNeXt V2 Training Pipeline
This notebook is optimized for a Google Colab T4 GPU (free tier) and implements a Multi-Head ConvNeXt V2 architecture for OCT Image Classification.

In [ ]:
!pip install timm monai grad-cam pandas scikit-learn

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from tqdm import tqdm
import timm
from monai.transforms import Compose, LoadImage, EnsureChannelFirst, Resize, ScaleIntensity

In [ ]:
# Head 2 mapping
H2_MAPPING = {
    'Macular_Degeneration': 0,
    'Diabetic_Complications': 1,
    'Vascular_Occlusions': 2,
    'Fluid_Accumulation': 3,
    'Structural_Issues': 4
}

# Head 3 mapping
H3_CLASSES = ['CNV', 'DRUSEN', 'Generic_AMD', 'DME', 'DR', 'MH', 'RVO', 'RAO', 'CSR', 'ERM', 'VID']
H3_MAPPING = {cls: idx for idx, cls in enumerate(H3_CLASSES)}

class MultiHeadOCTDataset(Dataset):
    def __init__(self, df_manifest, transforms=None):
        """
        df_manifest: pandas DataFrame with columns ['image_path', 'head1_label', 'head2_label', 'head3_labels']
                     where head3_labels is a list or comma-separated string of classes.
        """
        self.df = df_manifest.reset_index(drop=True)
        self.transforms = transforms
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image_path']
        
        # 1. Load and transform image
        if self.transforms:
            image = self.transforms(img_path)
        else:
            image = LoadImage(image_only=True)(img_path)
            
        # 2. Extract Head 1 Label (Binary: 0 or 1)
        h1_label = torch.tensor([float(row['head1_label'])], dtype=torch.float32)
        
        # 3. Extract Head 2 Label (Multi-class: 0-4)
        h2_str = row['head2_label']
        h2_label = torch.tensor(H2_MAPPING.get(h2_str, -1), dtype=torch.long)
        
        # 4. Extract Head 3 Label (Multi-label: 11 classes)
        h3_labels_raw = row['head3_labels']
        if isinstance(h3_labels_raw, str):
            h3_labels_raw = [x.strip() for x in h3_labels_raw.split(',')]
        elif isinstance(h3_labels_raw, float) and np.isnan(h3_labels_raw):
            h3_labels_raw = []
        
        h3_vector = torch.zeros(len(H3_CLASSES), dtype=torch.float32)
        for cls in h3_labels_raw:
            if cls in H3_MAPPING:
                h3_vector[H3_MAPPING[cls]] = 1.0
                
        return image, (h1_label, h2_label, h3_vector)

# Define MONAI Transforms
train_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((384, 384)),
    ScaleIntensity()
])

val_transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),
    Resize((384, 384)),
    ScaleIntensity()
])

In [ ]:
class MultiHeadConvNeXtV2(nn.Module):
    def __init__(self, model_name='convnextv2_base', pretrained=True):
        super(MultiHeadConvNeXtV2, self).__init__()
        
        # Load pre-trained ConvNeXt V2 (without classifier head)
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        embed_dim = self.backbone.num_features
        
        # Head 1: Binary Classification
        self.head1 = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )
        
        # Head 2: 5-class Classification
        self.head2 = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 5)
        )
        
        # Head 3: 11-class Multi-label Classification
        self.head3 = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 11)
        )
        
    def forward(self, x):
        features = self.backbone(x)
        out1 = self.head1(features)
        out2 = self.head2(features)
        out3 = self.head3(features)
        return out1, out2, out3
        
    def freeze_backbone(self, warmup=True):
        """
        warmup=True: Freeze backbone except the last stage (stages.3) and classification heads.
        warmup=False: Unfreeze all layers for fine-tuning.
        """
        if not warmup:
            for param in self.backbone.parameters():
                param.requires_grad = True
            print("Backbone completely unfrozen for fine-tuning.")
        else:
            for name, param in self.backbone.named_parameters():
                # ConvNeXt architectures have 'stages.3' as the final stage
                if 'stages.3' in name or 'norm' in name.split('.')[-1]:
                    param.requires_grad = True
                else:
                    param.requires_grad = False
            print("Backbone partially frozen (Stage 4 & MLPs unfrozen for warmup).")

In [ ]:
# Hyperparameters
BATCH_SIZE = 16 # Fit in 16GB T4 VRAM with AMP
NUM_EPOCHS_WARMUP = 5
NUM_EPOCHS_FINETUNE = 15
LEARNING_RATE_WARMUP = 1e-3
LEARNING_RATE_FINETUNE = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

MANIFEST_PATH = 'dataset_manifest.csv' # Set to your actual CSV path
import os
if os.path.exists(MANIFEST_PATH):
    print("Loading dataset from manifest...")
    df_train = pd.read_csv(MANIFEST_PATH)
else:
    print("Manifest not found. Creating dummy dataset for illustration...")
    df_train = pd.DataFrame({
        'image_path': ['dummy1.jpg', 'dummy2.jpg'] * 50,
        'head1_label': [1, 0] * 50,
        'head2_label': ['Macular_Degeneration', 'Diabetic_Complications'] * 50,
        'head3_labels': [['CNV', 'DRUSEN'], ['DME', 'DR']] * 50
    })

dataset = MultiHeadOCTDataset(df_train, transforms=train_transforms)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# Initialize Model
model = MultiHeadConvNeXtV2(model_name='convnextv2_base', pretrained=True).to(device)

# Loss Functions
criterion_h1 = nn.BCEWithLogitsLoss()
criterion_h2 = nn.CrossEntropyLoss()
criterion_h3 = nn.BCEWithLogitsLoss()

# Scaler for Automatic Mixed Precision (AMP)
scaler = torch.amp.GradScaler('cuda')

In [ ]:
def train_one_epoch(model, dataloader, optimizer, scaler, epoch_desc):
    model.train()
    running_loss = 0.0
    
    progress_bar = tqdm(dataloader, desc=epoch_desc)
    for images, (labels_h1, labels_h2, labels_h3) in progress_bar:
        images = images.to(device)
        labels_h1 = labels_h1.to(device)
        labels_h2 = labels_h2.to(device)
        labels_h3 = labels_h3.to(device)
        
        optimizer.zero_grad()
        
        # AMP context
        with torch.amp.autocast('cuda'):
            out1, out2, out3 = model(images)
            
            loss1 = criterion_h1(out1, labels_h1)
            loss2 = criterion_h2(out2, labels_h2)
            loss3 = criterion_h3(out3, labels_h3)
            
            # Combined Loss
            total_loss = loss1 + loss2 + loss3
            
        # Backward pass with scaler
        scaler.scale(total_loss).backward()
        
        # Gradient Clipping (max_norm = 1.0)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += total_loss.item()
        progress_bar.set_postfix({'loss': running_loss / (progress_bar.n + 1)})
        
    return running_loss / len(dataloader)

# ================================
# PHASE 1: Warm-up
# ================================
print("Starting Warm-up Phase...")
model.freeze_backbone(warmup=True)
optimizer_warmup = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE_WARMUP)

for epoch in range(NUM_EPOCHS_WARMUP):
    train_loss = train_one_epoch(model, dataloader, optimizer_warmup, scaler, f"Warmup Epoch {epoch+1}/{NUM_EPOCHS_WARMUP}")
    print(f"Epoch {epoch+1} Loss: {train_loss:.4f}")

# ================================
# PHASE 2: Fine-tuning
# ================================
print("\nStarting Fine-tuning Phase...")
model.freeze_backbone(warmup=False)
optimizer_finetune = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE_FINETUNE, weight_decay=1e-4)

for epoch in range(NUM_EPOCHS_FINETUNE):
    train_loss = train_one_epoch(model, dataloader, optimizer_finetune, scaler, f"Finetune Epoch {epoch+1}/{NUM_EPOCHS_FINETUNE}")
    print(f"Epoch {epoch+1} Loss: {train_loss:.4f}")

print("\nTraining complete! Saving checkpoint...")
torch.save(model.state_dict(), 'multi_head_convnextv2_best.pth')
print("Checkpoint saved to 'multi_head_convnextv2_best.pth'")